<a href="https://colab.research.google.com/github/dheerajdabbeti/dataviz-exercises-dheerajdabbeti/blob/main/lecture06_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 6 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week06/lecture06_exercise.ipynb`

**Rules:**
1. Use `px` first, then customise with `update_traces` / `update_layout`
2. Colour encodes a meaningful category — not decoration
3. Insight title names the specific finding
4. Consider: would a bar chart be clearer? If yes, use the bar chart

---


In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('../data/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [2]:
# Task 1 — Fossil Fuel Treemap

import pandas as pd
import plotly.express as px

# Load dataset
df = pd.read_csv('/content/global_energy_mix.csv')

fossil_sources = ['Coal', 'Oil', 'Natural Gas']
fossil_df = df[df['Source'].isin(fossil_sources)].copy()
top_region = (
    fossil_df.groupby('Region')['TWh']
    .sum()
    .idxmax()
)

# Insight-based title
title_text = f'{top_region} Is the Most Fossil Fuel Dependent Region'

fossil_colors = {
    'Coal': '#D55E00',        # orange-red
    'Oil': '#0072B2',         # blue
    'Natural Gas': '#009E73' # green
}

fig = px.treemap(
    fossil_df,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map=fossil_colors,
    title=title_text
)

fig.update_traces(
    textinfo='label+value',
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    root_color='lightgrey',
    marker=dict(
        line=dict(color='white', width=1)
    )
)

fig.update_layout(
    margin=dict(t=60, l=10, r=10, b=10),
    uniformtext=dict(minsize=10)
)
fig.show()


## Task 2 — Sunburst: tipping behaviour by day and meal time

**What to build:** A sunburst chart using the built-in `tips` dataset showing how **total bill amount** is distributed across day → time → smoker status.

**Requirements:**
- Load tips with `px.data.tips()`
- Aggregate **total bill** (sum of `total_bill`) per group — not count
- Hierarchy: `path=['day', 'time', 'smoker']`
- Colour encodes smoker status with a CVD-safe blue/orange palette
- Grey out parent nodes (day and time level)
- Use `percent parent` for text labels
- Insight title describing where the most spending happens

> 💡 `tips.groupby(['day', 'time', 'smoker'])['total_bill'].sum().reset_index()`


In [3]:
# Task 2 — Sunburst Chart: Tipping Behaviour by Day and Meal Time

import pandas as pd
import plotly.express as px
tips = px.data.tips()
tips_grouped = (
    tips.groupby(['day', 'time', 'smoker'])['total_bill']
    .sum()
    .reset_index()
)
top_spending = (
    tips_grouped.groupby(['day', 'time'])['total_bill']
    .sum()
    .reset_index()
    .sort_values(by='total_bill', ascending=False)
    .iloc[0]
)

title_text = (
    f'Highest Spending Happens on {top_spending["day"]} '
    f'During {top_spending["time"]}'
)
smoker_colors = {
    'Yes': '#0072B2',  # blue
    'No': '#E69F00'    # orange
}
fig = px.sunburst(
    tips_grouped,
    path=['day', 'time', 'smoker'],
    values='total_bill',
    color='smoker',
    color_discrete_map=smoker_colors,
    title=title_text
)
fig.update_traces(
    textinfo='label+percent parent',
    insidetextorientation='radial',
    marker=dict(
        line=dict(color='white', width=1)
    ),
    root_color='lightgrey'
)
fig.update_layout(
    margin=dict(t=60, l=10, r=10, b=10),
    uniformtext=dict(minsize=10)
)
fig.show()

## Task 3 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country


In [4]:
import pandas as pd
import plotly.express as px

df = pd.read_csv('/content/global_energy_mix.csv')
low_carbon_sources = ['Hydro', 'Nuclear']

low_carbon_df = df[df['Source'].isin(low_carbon_sources)].copy()
country_totals = (
    low_carbon_df.groupby('Country')['TWh']
    .sum()
    .reset_index()
)
country_totals['All'] = 'Low-carbon'
top_country = (
    country_totals.sort_values(by='TWh', ascending=False)
    .iloc[0]['Country']
)
bar_color = '#0072B2'  # blue
treemap_fig = px.treemap(
    country_totals,
    path=['All', 'Country'],
    values='TWh',
    color='TWh',
    color_continuous_scale='Blues',
    title='Low-Carbon Energy Distribution by Country'
)
treemap_fig.update_traces(
    textinfo='label+value',
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    root_color='lightgrey',
    marker=dict(
        line=dict(color='white', width=1)
    )
)
treemap_fig.update_layout(
    margin=dict(t=60, l=10, r=10, b=10),
    uniformtext=dict(minsize=10)
)

# Display treemap
treemap_fig.show()
bar_fig = px.bar(
    country_totals.sort_values(by='TWh'),
    x='TWh',
    y='Country',
    orientation='h',
    text='TWh',
    color_discrete_sequence=[bar_color],

    # Insight title
    title=f'{top_country} Leads Low-Carbon Energy Production'
)
bar_fig.update_traces(
    texttemplate='%{text:,.0f} TWh',
    textposition='outside'
)

bar_fig.update_layout(
    xaxis_title='Total Low-Carbon Energy (TWh)',
    yaxis_title='Country',
    margin=dict(t=60, l=10, r=10, b=10),
    showlegend=False
)

# Display bar chart
bar_fig.show()
